In [235]:
import numpy as np
import pandas as pd
import scipy
import re
import os
from pathlib import Path

Raw datasets downloaded from https://mjl.clarivate.com/collection-list-downloads

on 13 May 2026

(according to WoS, last updated 20 April 2026)

In [236]:
data_path = Path('../data/260513_wos_cc/')
os.listdir(data_path)
#pd.read_csv()

['Arts & Humanities Citation Index (AHCI).csv',
 'Emerging Sources Citation Index (ESCI).csv',
 'JCR 2025.csv',
 'JOURNAL_CHANGE_MONTH_2026_01.xlsx',
 'JOURNAL_CHANGE_MONTH_2026_02.xlsx',
 'JOURNAL_CHANGE_MONTH_2026_03.xlsx',
 'JOURNAL_CHANGE_MONTH_2026_04.xlsx',
 'JOURNAL_CHANGE_YEAR_2023.xlsx',
 'JOURNAL_CHANGE_YEAR_2024.xlsx',
 'JOURNAL_CHANGE_YEAR_2025.xlsx',
 'Science Citation Index Expanded (SCIE).csv',
 'Social Sciences Citation Index (SSCI).csv']

In [237]:
# current collection

collect = []
for filename, collection in zip(['Arts & Humanities Citation Index (AHCI).csv',
 'Emerging Sources Citation Index (ESCI).csv',
 'Science Citation Index Expanded (SCIE).csv',
 'Social Sciences Citation Index (SSCI).csv'], 
                    ['AHCI', 'ESCI', 'SCIE', 'SSCI']):
    df = pd.read_csv(str(data_path) + '/' + filename, encoding='utf-8')
    df['collection'] = collection
    collect.append(df)
df_current = pd.concat(collect)

In [238]:
# past collection

collect = []
for filename in ['JOURNAL_CHANGE_YEAR_2023.xlsx',
 'JOURNAL_CHANGE_YEAR_2024.xlsx',
 'JOURNAL_CHANGE_YEAR_2025.xlsx',
                 'JOURNAL_CHANGE_MONTH_2026_01.xlsx',
 'JOURNAL_CHANGE_MONTH_2026_02.xlsx',
 'JOURNAL_CHANGE_MONTH_2026_03.xlsx',
 'JOURNAL_CHANGE_MONTH_2026_04.xlsx']:
    df = pd.read_excel(str(data_path) + '/' + filename)
    df['source_filename'] = filename
    collect.append(df)
df_changes = pd.concat(collect)
df_changes.columns = list(pd.read_excel(str(data_path) + '/' + 'JOURNAL_CHANGE_YEAR_2025.xlsx', header=27).columns) + ['source_filename']
df_changes = df_changes.dropna(subset=['Journal title'])

prohib = ['Journal title',
          'The Master Journal List is available online here ', 
          'Details of our journal evaluation process and selection criteria are available here']
df_changes = df_changes.loc[~df_changes['Journal title'].isin(prohib)].copy()

In [239]:
# combine

df_current['indexing_status'] = 'currently indexed'
df_current['notes_on_indexing_status'] = df_current['collection']
df_changes_slice = df_changes[df_changes['Coverage change'] == 'Editorial De-listing'].copy()
df_changes_slice['indexing_status'] = 'deindexed'
df_changes_slice['notes_on_indexing_status'] = df_changes_slice['source_filename']

df_all = pd.concat([df_current, df_changes_slice])

In [240]:
df_all['journal_title'] = df_all['Journal title']
df_all['p_issn'] = df_all['ISSN']
df_all['e_issn'] = df_all['eISSN']
df_all['service'] = 'wos' 
df_all['internal_identifier'] = None

In [241]:
df_all_slice = df_all[['service', 'internal_identifier', 'journal_title', 'p_issn', 'e_issn', 'indexing_status', 'notes_on_indexing_status']].copy()

In [242]:
df_all_slice.groupby('journal_title')['indexing_status'].nunique().sort_values()

journal_title
1616-ANUARIO DE LITERATURA COMPARADA    1
LIBRARY & INFORMATION HISTORY           1
LIBRARY                                 1
LIBERTE                                 1
LIBERABIT-REVISTA DE PSICOLOGIA         1
                                       ..
FOURRAGES                               1
FOUNDATIONS OF SCIENCE                  1
FRANCAIS MODERNE                        1
ZYGOTE                                  1
BIOMED RESEARCH INTERNATIONAL           2
Name: indexing_status, Length: 23149, dtype: int64

In [243]:
file_prefix = '260513_wos_cc'
df_all_slice.to_csv('../data/' + file_prefix + '.csv', index=False)
df_all_slice.to_parquet('../data/' + file_prefix + '.parquet')